In [1]:
import dascore as dc
import matplotlib.pyplot as plt
from dascore.units import Hz
import numpy as np
import os

# --- Configuration ---
target_dist = 17500
file_path = "das_records/good-events-3.2-up/ak0239vxdtm6_TERRA.h5"
output_file = "terra_single_channel_black.pdf"

try:
    # 1. Load the data
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Could not find {file_path}")

    spool = dc.spool(file_path)
    patch = spool[0]
    
    # 2. Pre-process (Filter to remove DC drift/low freq noise)
    # This matches the (1 * Hz, None) filter from your uberfigure
    processed_patch = patch.pass_filter(time=(1 * Hz, None))
    
    # 3. Find the specific channel index for 17,500m
    distances = processed_patch.coords.get_array('distance')
    channel_index = np.argmin(np.abs(distances - target_dist))
    actual_dist = distances[channel_index]
    
    # 4. Extract the time-series data
    # Data shape is typically (time, distance)
    trace_data = processed_patch.data[:, channel_index]
    times = processed_patch.coords.get_array('time')

    # 5. Plotting
    fig, ax = plt.subplots(figsize=(15, 5))
    
    # Plotting as a standard black line (seismogram style)
    ax.plot(times, trace_data, color='black', linewidth=0.8)

    timebig = 19572.91452
    # Formatting
    ax.set_xlabel("Time (hh-mm-ss)", fontsize=12)
    ax.set_ylabel("Strain (Radians)", fontsize=12)
    plt.xlim(timebig, (timebig + 0.00145))
    plt.tight_layout()
    plt.savefig(output_file)
    plt.show()
    
    print(f"Successfully plotted channel {channel_index} to {output_file}")

except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: Could not find das_records/good-events-3.2-up/ak0239vxdtm6_TERRA.h5


In [2]:
import dascore as dc
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as patches
from dascore.units import Hz
import numpy as np
import os

def matplotlib_wiggle(ax, data, times, dist_coords, scale, linewidth, alpha, highlight_dist=None):
    """
    Plots a DAS patch as a wiggle plot, optionally highlighting one trace in red.
    """
    if data.size == 0: 
        return

    n_samples, n_channels = data.shape
    
    # Identify the index to highlight red
    highlight_idx = None
    if highlight_dist is not None:
        highlight_idx = np.argmin(np.abs(dist_coords - highlight_dist))

    global_max = np.nanpercentile(np.abs(data), 99)
    if global_max <= 0 or not np.isfinite(global_max): global_max = 1.0
    
    for i in range(n_channels):
        trace = data[:, i]
        if np.all(np.isnan(trace)): continue
                
        scaled_trace = (np.nan_to_num(trace) / global_max) * scale
        
        # Determine color and style
        if i == highlight_idx:
            color, lw, l_alpha = ('red', linewidth * 1.5, 1.0)
        else:
            color, lw, l_alpha = ('black', linewidth, alpha)

        ax.plot(times, i + scaled_trace, color=color, linewidth=lw, alpha=l_alpha)
    
    ax.set_ylim(-1, n_channels)

def plot_zoomed_wiggle(patch, ax, title, highlight_dist=17500):
    """
    Processes and plots an UNDECIMATED DAS patch with a red highlight at 17,500m.
    """
    try:
        process1_patch = patch.dropna(dim="time", how="all")
        processed_patch = process1_patch.pass_filter(time=(1 * Hz, None))
        data = processed_patch.data
        times = processed_patch.coords.get_array('time')
        dists = processed_patch.coords.get_array('distance')
        
        # Pass distance array and highlight target to the wiggle function
        matplotlib_wiggle(ax, data, times, dists, scale=2, linewidth=3, alpha=0.5, highlight_dist=highlight_dist)         
        ax.set_title(title, fontsize=10)
        ax.invert_yaxis()
        
    except Exception as e:
        print(f"Failed to plot wiggle for {title}: {e}")
        ax.set_title(f"Failed to plot: {title}")

def set_meter_ticks(ax, dist_coords, tick_interval_m=1000):
    try:
        min_dist = np.min(dist_coords)
        max_dist = np.max(dist_coords)
        start_m = np.ceil(min_dist / tick_interval_m) * tick_interval_m
        end_m = np.floor(max_dist / tick_interval_m) * tick_interval_m
        meter_labels = np.arange(start_m, end_m + tick_interval_m, tick_interval_m)
        tick_indices = [np.argmin(np.abs(dist_coords - m)) for m in meter_labels]
        ax.set_yticks(tick_indices)
        ax.set_yticklabels([f"{int(m)}" for m in meter_labels])
        ax.set_ylabel("Distance (m)", fontsize=12)
    except Exception as e:
        ax.set_ylabel("Channel Index") 

# --- Directory and Path Setup ---
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

data_dir = os.path.normpath(os.path.join(script_dir, "das_records", "good-events-3.2-up"))
output_dir = os.path.normpath(os.path.join(script_dir, "das_figures"))
os.makedirs(output_dir, exist_ok=True)

terra_file_path = os.path.join(data_dir, "ak0239vxdtm6_TERRA.h5")
output_file = os.path.join(output_dir, "uberfigure_terra_highlighted.pdf")

if not os.path.exists(terra_file_path):
    print(f"ERROR: File not found at {os.path.abspath(terra_file_path)}")
else:
    try:
        terra_spool = dc.spool(terra_file_path)
        terra_patch = terra_spool[0]
        
        terra_subsets = {
            "Dist_17.5k_Center": {'dist': (17400, 17600), 'P-Arrival': (9, 14)}
        }

        ref_plot_height_inches = 40
        zoom_plot_height_inches = 40 
        num_terra_zooms = sum(len(v) - 1 for v in terra_subsets.values()) 
        num_total_plots = 1 + num_terra_zooms
        height_ratios = [ref_plot_height_inches] + [zoom_plot_height_inches] * num_terra_zooms
        
        fig, axes = plt.subplots(num_total_plots, 1, figsize=(20, sum(height_ratios)), 
                                 gridspec_kw={'height_ratios': height_ratios})

        if num_total_plots == 1: axes = [axes]
        
        ax_ref = axes[0]
        t_start = terra_patch.coords.min('time')
        
        # 1. Reference Plot
        processed_ref = (terra_patch.decimate(distance=4).pass_filter(time=(1 * Hz, None)))
        ref_dists = processed_ref.coords.get_array('distance')
        
        matplotlib_wiggle(ax_ref, processed_ref.data, processed_ref.coords.get_array('time'), 
                          ref_dists, scale=5, linewidth=0.5, alpha=0.3)
        
        ax_ref.set_title(f"TERRA Reference Plot: {os.path.basename(terra_file_path)}", fontsize=26)
        set_meter_ticks(ax_ref, ref_dists, 2000)
        ax_ref.invert_yaxis()

        # 2. Zoom Plotting with Box logic
        current_plot_index = 1
        for subset_name, subset_info in terra_subsets.items():
            dist_range = subset_info['dist']
            for event_name, (start_sec, end_sec) in subset_info.items():
                if event_name == 'dist': continue 
                
                t_box_start = t_start + np.timedelta64(start_sec, 's')
                t_box_end = t_start + np.timedelta64(end_sec, 's')
                
                # Draw the Red Box on the Reference Plot
                y_low = np.argmin(np.abs(ref_dists - dist_range[0]))
                y_high = np.argmin(np.abs(ref_dists - dist_range[1]))
                rect = patches.Rectangle((t_box_start, y_low), t_box_end - t_box_start, y_high - y_low,
                                         linewidth=6, edgecolor='red', facecolor='none', zorder=10)
                ax_ref.add_patch(rect)

                # Generate the Zoom Plot
                ax_zoom = axes[current_plot_index]
                zoomed_patch = terra_patch.select(time=(t_box_start, t_box_end), distance=dist_range)
                
                title = (f"Zoom (TERRA): {event_name}\n"
                         f"Distance: {dist_range[0]}m - {dist_range[1]}m | Time: {start_sec}s to {end_sec}s")
                plot_zoomed_wiggle(zoomed_patch, ax_zoom, title, highlight_dist=17500)
                
                current_plot_index += 1

        plt.savefig(output_file, bbox_inches='tight')
        plt.close(fig)
        print(f"Successfully saved to {output_file}")

    except Exception as e:
        print(f"Error: {e}")

ERROR: File not found at /Users/ed/research_code/das/minor_das_scripts/das_records/good-events-3.2-up/ak0239vxdtm6_TERRA.h5
